## Базовый пример запуска Gamac

### Импортируем нужные библиотеки

In [1]:
import sys
sys.path.append('../')

import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from torchvision.datasets import CIFAR100

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### Получим данные для кластеризации

In [2]:
# Импортируем датафрейм для табличных данных
data = load_digits(as_frame=True)

table = data['data']
table.head()

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_6_6,pixel_6_7,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,9.0,0.0,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0


In [3]:
# Импортируем данные для CIFAR
# Первая загрузка будет долгая так как он грузит из open-source
cifar100 = CIFAR100('../data/cifar', download=True, train=False)

cifar_txt = [f'a photo of {cifar100.classes[img[1]]}' for img in cifar100][:100]
cifar_img = [img[0] for img in cifar100][:100]
cifar_table = pd.DataFrame(cifar100.targets[:100])

### Инициализируем Gamac
- Класс Gamac имеет следующие аргументы:
    - mab_solver: MabSolvers = MabSolvers.SOFTMAX - тип алгоритма multi-arm bandit
    - hyper_optimiser: HyperOptimisers = HyperOptimisers.OPTUNA - Тип оптимизатора, по умолчанию Optuna
    - target_measures: list[str] | None = None - Целевая мера, по умолчанию не указывается - в этом случае происходит автоматический подбор меры
    - time_limit: int | None = None - Время в секундах поиска оптимальной модели кластеризации
    - iter_limit: int | None = 50 - Кол-во итераций поиска оптимальной модели кластеризации
    - batch_size: int = 32 - размер батча для предобработки текста и изображений
    - model_name: str = "openai/clip-vit-large-patch14" - CLIP-based модель для обработки изображений и текста

In [4]:
# По умолчанию кол-во итераций стоит 50, в реальных кейсах число лучше ставить больше
gamac = Gamac(iter_limit=30)

### Запустим поиск оптимального алгоритма кластеризации для табличных данных
Работа происходит в три этапа
1. Обработка и приведение модальностей в единый вектор
2. Определение меры - можно задать явно в target_measures, либо оставить пустым (в этом случае мера будет подобрана автоматически)
3. Поиск алгоритма кластеризации с наилучшими по выбранным мерам гиперпараметрами
- В ходе перебора логируются успешные и неуспешные запуски перебора
- Неуспешные запуски появляются в ходе получения некорректных гиперпараметров алгоритма под конкретный датасет

### Базовый пример для табличных данных

In [5]:
dataset, best_model = gamac.run(table=table)

=== Started CVI prediction ===
=== CVI prediction iteration 1/8 ====
=== CVI prediction iteration 2/8 ====
=== CVI prediction iteration 3/8 ====
=== CVI prediction iteration 4/8 ====
=== CVI prediction iteration 5/8 ====
=== CVI prediction iteration 6/8 ====
=== CVI prediction iteration 7/8 ====
=== CVI prediction iteration 8/8 ====
=== Picked SYM in 11.440877914428711s ===
=== MEASURES 0.0498964786529541s, {'SYM': 7.769545261478631e-06} ===
=== MEASURES 0.025429725646972656s, {'SYM': 9.260516945468355e-05} ===
=== ALGO 0.21842432022094727s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 2, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 5.5679099559783936s, FailedRun, {'bandwidth': 0.09685207693168384, 'max_iter': 57, 'tol': 9.354686872031158e-05} ===
=== ALGO 1.0208547115325928s, FailedRun, {'name': 'DBSCAN', 'eps': 0.08611323149440304, 'eps_sq': 0.0074154886384086485, 'min_samples': 6} ===
=== ALGO 5.324005603790283s, FailedRun, {'min_cluster_size': 8, 'min_

### На выходе получаем итоговый датасет после обработки

In [6]:
print(dataset.shape)
print(dataset[:10])

(1797, 64)
[[ 0.0000000e+00 -3.3501649e-01 -4.3081019e-02  2.7407151e-01
  -6.6447753e-01 -8.4412938e-01 -4.0972391e-01 -1.2502292e-01
  -5.9077557e-02 -6.2400925e-01  4.8297450e-01  7.5962245e-01
  -5.8425862e-02  1.1277211e+00  8.7958306e-01 -1.3043338e-01
  -4.4625074e-02  1.1144272e-01  8.9588046e-01 -8.6066633e-01
  -1.1496484e+00  5.1547188e-01  1.9059634e+00 -1.1422184e-01
  -3.3379726e-02  4.8648927e-01  4.6988511e-01 -1.4999014e+00
  -1.6140628e+00  7.6397777e-02  1.5418141e+00 -4.7232382e-02
   0.0000000e+00  7.6465553e-01  5.2630186e-02 -1.4476300e+00
  -1.7366644e+00  4.3615878e-02  1.4395580e+00  0.0000000e+00
  -6.1343670e-02  8.1055361e-01  6.3011712e-01 -1.1224571e+00
  -1.0662316e+00  6.6096473e-01  8.1845075e-01 -8.8741615e-02
  -3.5433263e-02  7.4211895e-01  1.1506522e+00 -8.6867058e-01
   1.1012974e-01  5.3761119e-01 -7.5743580e-01 -2.0978513e-01
  -2.3596458e-02 -2.9908136e-01  8.6718693e-02  2.0829257e-01
  -3.6677122e-01 -1.1466475e+00 -5.0566983e-01 -1.9600752e-

### Также получаем информацию о лучшей модели - ее наилучшей конфигурацией и финального значения мер

In [7]:
print(best_model.estimation)

{<Internal.SYM: ('sym', <function sym at 0x7865e39d6160>)>: 0.004690165951458465}


### Получение меток по лучшему классификатору

In [8]:
print(best_model.model.predict(dataset))

[12  9  9 ...  6  6  6]


### Пример для мультимодальных данных
Для этого используем данные по CIFAR100

### В данном примере можно использовать все модальности (таблица, текст и изображение)

In [9]:
# В данном примере можно использовать все модальности (таблица, текст и изображение)
gamac = Gamac(iter_limit=30, target_measures={Internal.SYM})
dataset, best_model = gamac.run(table=cifar_table, text=cifar_txt, image=cifar_img)

print(best_model.model.predict(dataset))

=== MEASURES 0.0028984546661376953s, {'BR': -0.9116096333724001} ===
=== ALGO 0.3218817710876465s, FailedRun, {'name': 'BisectingKMeans', 'n_clusters': 10, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 0.3848719596862793s, FailedRun, {'bandwidth': 0.6879599100889358, 'max_iter': 286, 'tol': 7.17707591272935e-05} ===
=== ALGO 0.0610504150390625s, FailedRun, {'name': 'DBSCAN', 'eps': 0.891537434307885, 'eps_sq': 0.7948389967722864, 'min_samples': 12} ===
=== ALGO 0.04186677932739258s, FailedRun, {'min_cluster_size': 13, 'min_samples': 7} ===
=== ALGO 0.7498795986175537s, FailedRun, {'threshold': 0.8334892251301127, 'branching_factor': 46, 'n_clusters': 11} ===
=== MEASURES 0.0022788047790527344s, {'BR': -0.4710360476541114} ===
=== ALGO 0.011594533920288086s, SuccessRun, {'name': 'KMeans', 'n_clusters': 3, 'max_iter': 100, 'tol': 0.0001, 'random_state': None} ===
=== ALGO 0.41548657417297363s, FailedRun, {'name': 'BisectingKMeans', 'n_clusters': 11, 'max_iter': 100, 'ini

### или использовать только текст + изображение

In [10]:
dataset, best_model = gamac.run(text=cifar_txt, image=cifar_img)

print(best_model.model.predict(dataset))

=== MEASURES 0.002105236053466797s, {'BR': -0.40597935529909646} ===
=== MEASURES 0.0023653507232666016s, {'BR': -0.39562565531878646} ===
=== ALGO 0.02897334098815918s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 3, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 0.16148018836975098s, FailedRun, {'bandwidth': 0.01231709847353924, 'max_iter': 112, 'tol': 4.080841540611773e-05} ===
=== ALGO 0.05630755424499512s, FailedRun, {'name': 'DBSCAN', 'eps': 0.5238506695308877, 'eps_sq': 0.27441952396795927, 'min_samples': 15} ===
=== ALGO 0.20755839347839355s, FailedRun, {'min_cluster_size': 10, 'min_samples': 10} ===
=== ALGO 0.7600841522216797s, FailedRun, {'threshold': 0.7532058894246878, 'branching_factor': 71, 'n_clusters': 9} ===
=== ALGO 0.03581428527832031s, FailedRun, {'name': 'KMeans', 'n_clusters': 13, 'max_iter': 100, 'tol': 0.0001, 'random_state': None} ===
=== MEASURES 0.0016875267028808594s, {'BR': -0.3969135593066042} ===
=== ALGO 0.013195037841796875s

### Таким образом Gamac может подбирать кластеризацию для:
1. табличных
2. изображений с текстом
3. совместно